In [63]:
# %% Cell 1 — Imports & Setup
# Purpose: Load libraries, set paths, define constants

import os, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from rdkit import Chem
from rdkit.Chem import Descriptors

# Paths tailored to your project
BASE_DIR = "/home/rohit/Desktop/NeurIPS"
ARTIFACT_DIR = os.path.join(BASE_DIR, "artifacts")
FEAT_DIR = os.path.join(BASE_DIR, "artifacts/features")
COMP_DIR = os.path.join(BASE_DIR, "competition")

# Load train/test
train = pd.read_csv(os.path.join(COMP_DIR, "train.csv"))
test = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))


targets = ["Tg", "FFV", "Tc", "Density", "Rg"]

print("✅ Imports complete")
print("Train shape:", train.shape, "Test shape:", test.shape)
print("Sample submission preview:\n", sample.head())


✅ Imports complete
Train shape: (7973, 7) Test shape: (3, 2)
Sample submission preview:
            id  Tg  FFV  Tc  Density  Rg
0  1109053969   0    0   0        0   0
1  1422188626   0    0   0        0   0
2  2032016830   0    0   0        0   0


In [64]:
# %% Cell — Load folds.json properly
import json

folds_path = os.path.join(ARTIFACT_DIR, "ingest", "folds.json")

with open(folds_path, "r") as f:
    raw = json.load(f)

raw_folds = raw["folds"]

# Convert to (train_idx, val_idx) pairs
all_idx = set(range(len(train)))
folds = []
for val_idx in raw_folds:
    val_idx = list(map(int, val_idx))
    train_idx = list(all_idx - set(val_idx))
    folds.append((train_idx, val_idx))

print(f"✅ Loaded {len(folds)} folds")
for i, (tr, va) in enumerate(folds):
    print(f"Fold {i}: train={len(tr)}, val={len(va)}")


✅ Loaded 5 folds
Fold 0: train=6008, val=1965
Fold 1: train=6698, val=1275
Fold 2: train=5614, val=2359
Fold 3: train=6672, val=1301
Fold 4: train=6912, val=1061


In [104]:
# %% Cell 2 — Load Feature Blocks
# Purpose: Load precomputed features (tabular, rdkit, mordred).
# Physics features: take from train.csv, fill NaN for test since Kaggle hides them.

feat_blocks = {
    "tabular": pd.read_parquet(os.path.join(FEAT_DIR, "train_tabular.parquet")),
    "rdkit": pd.read_parquet(os.path.join(FEAT_DIR, "train_rd.parquet")),
    "mordred": pd.read_parquet(os.path.join(FEAT_DIR, "train_mordred.parquet"))
}
feat_blocks_test = {
    "tabular": pd.read_parquet(os.path.join(FEAT_DIR, "test_tabular.parquet")),
    "rdkit": pd.read_parquet(os.path.join(FEAT_DIR, "test_rd.parquet")),
    "mordred": pd.read_parquet(os.path.join(FEAT_DIR, "test_mordred.parquet"))
}

# Align indices
for k in feat_blocks:
    feat_blocks[k].reset_index(drop=True, inplace=True)
    feat_blocks_test[k].reset_index(drop=True, inplace=True)

# Physics cols only exist in train.csv, not in test.csv
physics_cols = ["Tg", "FFV", "Tc", "Density", "Rg"]
feat_blocks["physics"] = train[physics_cols].copy()

feat_blocks_test["physics"] = pd.DataFrame(
    np.nan, index=range(len(test)), columns=physics_cols
)

print("✅ Feature blocks loaded")
for k, v in feat_blocks.items():
    print(f"{k} | train: {v.shape}, test: {feat_blocks_test[k].shape}")


✅ Feature blocks loaded
tabular | train: (7961, 3469), test: (3, 3469)
rdkit | train: (7961, 2057), test: (3, 2057)
mordred | train: (7961, 1412), test: (3, 1412)
physics | train: (7973, 5), test: (3, 5)


In [105]:
# %% Cell 3 — Merge Feature Blocks + Deduplicate Columns
# Purpose: Combine all feature sources into single train/test DataFrames.
# - Includes tabular, rdkit, mordred, physics (from train.csv)
# - Aligns indices
# - Forces globally unique column names to avoid LightGBM errors

def make_unique_columns(df):
    """Ensure all column names are unique by appending suffixes to duplicates."""
    seen = {}
    new_cols = []
    for col in df.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    df.columns = new_cols
    return df

# Merge all blocks
features_train = pd.concat([
    feat_blocks["tabular"],
    feat_blocks["rdkit"],
    feat_blocks["mordred"],
    feat_blocks["physics"]
], axis=1)

features_test = pd.concat([
    feat_blocks_test["tabular"],
    feat_blocks_test["rdkit"],
    feat_blocks_test["mordred"],
    feat_blocks_test["physics"]
], axis=1)

# Deduplicate column names
features_train = make_unique_columns(features_train)
features_test = make_unique_columns(features_test)

# Align with train/test index lengths
features_train.reset_index(drop=True, inplace=True)
features_test.reset_index(drop=True, inplace=True)

print("✅ Features merged and deduplicated")
print("Train features:", features_train.shape)
print("Test features:", features_test.shape)
print("Example cols:", features_train.columns[:10].tolist())


✅ Features merged and deduplicated
Train features: (7973, 6943)
Test features: (3, 6943)
Example cols: ['nAcid', 'nBase', 'SpAbs_A', 'SpMax_A', 'SpDiam_A', 'SpAD_A', 'SpMAD_A', 'LogEE_A', 'VE1_A', 'VE2_A']


In [106]:
# %% Cell 4 — Align Features with Targets (Fixed)
# Purpose: Ensure features_train matches y_train rows and inspect missing targets.
# - Aligns by row order instead of index labels

# Reset index for both to enforce positional alignment
features_train = features_train.reset_index(drop=True)
features_test = features_test.reset_index(drop=True)
y_train_aligned = train[targets].reset_index(drop=True)

# Sanity check
print("✅ Alignment complete")
print("Features shape:", features_train.shape)
print("Targets shape:", y_train_aligned.shape)
print("\nMissing target counts:")
print(y_train_aligned.isna().sum())


✅ Alignment complete
Features shape: (7973, 6943)
Targets shape: (7973, 5)

Missing target counts:
Tg         7462
FFV         943
Tc         7236
Density    7360
Rg         7359
dtype: int64


In [107]:
# %% Cell 5 — Baseline Per-Target LightGBM Training (Fixed)
# Purpose: Train one LightGBM model per target with 5-fold CV
# - Converts all features to numeric
# - Handles missing labels by masking per target
# - Reports out-of-fold MAE for each property
# - Saves OOF preds for later stacking

from sklearn.metrics import mean_absolute_error
from lightgbm import early_stopping, log_evaluation

# ✅ Ensure features are numeric
features_train_num = features_train.apply(pd.to_numeric, errors="coerce").fillna(0).astype(np.float32)
features_test_num  = features_test.apply(pd.to_numeric, errors="coerce").fillna(0).astype(np.float32)

print("Sanity check:", features_train_num.shape, features_test_num.shape)
print("Dtypes after conversion:", features_train_num.dtypes.unique())

# Storage for OOF and test preds
oof_preds = pd.DataFrame(index=features_train.index, columns=targets)
test_preds = pd.DataFrame(index=features_test.index, columns=targets)

for t in targets:
    print(f"\n===== Target: {t} =====")
    mask = ~y_train_aligned[t].isna()        # only keep rows with labels
    X_t, y_t = features_train_num.loc[mask], y_train_aligned.loc[mask, t]
    
    oof_t = pd.Series(index=X_t.index, dtype=float)
    test_t = np.zeros((features_test_num.shape[0], len(folds)))
    
    for fold_n, (tr_idx, va_idx) in enumerate(folds):
        tr_idx_t = list(set(tr_idx) & set(X_t.index))
        va_idx_t = list(set(va_idx) & set(X_t.index))
        
        if len(va_idx_t) == 0:
            continue
        
        X_tr, y_tr = X_t.loc[tr_idx_t], y_t.loc[tr_idx_t]
        X_va, y_va = X_t.loc[va_idx_t], y_t.loc[va_idx_t]
        
        model = lgb.LGBMRegressor(
            n_estimators=5000,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="mae",
            callbacks=[early_stopping(50), log_evaluation(100)]
        )
        
        oof_t.loc[va_idx_t] = model.predict(X_va)
        test_t[:, fold_n] = model.predict(features_test_num)
    
        # Store OOF and test predictions
    oof_preds[t] = oof_t
    test_preds[t] = test_t.mean(axis=1)

    # ✅ Align indices before evaluating
    valid_idx = y_t.index.intersection(oof_t.dropna().index)
    mae = mean_absolute_error(y_t.loc[valid_idx], oof_t.loc[valid_idx])
    print(f"✅ OOF MAE for {t}: {mae:.5f}")


print("\n✅ Baseline LightGBM complete")
print("OOF preds shape:", oof_preds.shape)
print("Test preds shape:", test_preds.shape)


Sanity check: (7973, 6943) (3, 6943)
Dtypes after conversion: [dtype('float32')]

===== Target: Tg =====
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 12.2808	valid_0's l2: 338.178
[200]	valid_0's l1: 10.1855	valid_0's l2: 246.123
Early stopping, best iteration is:
[193]	valid_0's l1: 10.1637	valid_0's l2: 245.492
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 12.9622	valid_0's l2: 583.782
[200]	valid_0's l1: 10.227	valid_0's l2: 422.824
Early stopping, best iteration is:
[165]	valid_0's l1: 10.1631	valid_0's l2: 417.477
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 17.1746	valid_0's l2: 566.971
[200]	valid_0's l1: 13.6515	valid_0's l2: 350.165
[300]	valid_0's l1: 13.4029	valid_0's l2: 338.024
[400]	valid_0's l1: 13.3638	valid_0's l2: 336.843
Early stopping, best iteration is:
[371]	valid_0's l1: 13.3236	valid_0's l2: 334.974
Training until validation scores don't improve for 50 roun

In [108]:
# %% Cell 6 — Collect and Summarize OOF Results (NaN-safe)
# Purpose: Aggregate OOF predictions, compute MAE per target, and display summary.

results = {}

for t in targets:
    # restrict to rows where target is known
    mask = ~y_train[t].isna()
    y_true = y_train.loc[mask, t]

    # restrict predictions to same index
    y_pred = oof_preds[t].loc[y_true.index]

    # drop rows with NaNs in predictions
    valid_idx = y_pred.dropna().index.intersection(y_true.index)
    if len(valid_idx) == 0:
        print(f"⚠️ Skipping {t}: no valid aligned rows")
        continue

    mae = mean_absolute_error(y_true.loc[valid_idx], y_pred.loc[valid_idx])
    results[t] = mae
    print(f"Target {t}: OOF MAE = {mae:.5f} | n={len(valid_idx)}")

print("\n===== Overall Results =====")
for k, v in results.items():
    print(f"{k}: {v:.5f}")
print("Mean MAE:", np.mean(list(results.values())))


Target Tg: OOF MAE = 10.20196 | n=420
Target FFV: OOF MAE = 0.00136 | n=7012
Target Tc: OOF MAE = 0.00711 | n=733
Target Density: OOF MAE = 0.01853 | n=611
Target Rg: OOF MAE = 0.40558 | n=612

===== Overall Results =====
Tg: 10.20196
FFV: 0.00136
Tc: 0.00711
Density: 0.01853
Rg: 0.40558
Mean MAE: 2.126908520536193


In [110]:
# %% Cell X — Final Evaluation (robust against NaNs)
# Purpose: Compute per-target MAEs and the competition-weighted MAE (wMAE)

from sklearn.metrics import mean_absolute_error

results = {}
total_error, total_count = 0.0, 0

for t in targets:
    # rows where target is observed
    mask = ~y_train[t].isna()
    idx = y_train.index[mask].intersection(oof_preds[t].index)

    if len(idx) == 0:
        continue

    y_true = y_train.loc[idx, t]
    y_pred = oof_preds[t].loc[idx]

    # drop NaNs in predictions
    valid_mask = ~y_pred.isna()
    y_true = y_true.loc[valid_mask]
    y_pred = y_pred.loc[valid_mask]

    if len(y_true) == 0:
        continue

    mae = mean_absolute_error(y_true, y_pred)
    results[t] = (mae, len(y_true))

    # accumulate weighted error
    total_error += mae * len(y_true)
    total_count += len(y_true)

# print per-target results
print("===== Per-target OOF MAE =====")
for t, (mae, n) in results.items():
    print(f"{t:8s}: MAE = {mae:.5f} | n={n}")

# compute weighted metric
wmae = total_error / total_count
print("\n===== Overall Competition Metric =====")
print(f"wMAE (competition scale): {wmae:.5f}")


===== Per-target OOF MAE =====
Tg      : MAE = 10.20196 | n=420
FFV     : MAE = 0.00136 | n=7012
Tc      : MAE = 0.00711 | n=733
Density : MAE = 0.01853 | n=611
Rg      : MAE = 0.40558 | n=612

===== Overall Competition Metric =====
wMAE (competition scale): 0.48563


In [111]:
# %% Cell — Correct competition wMAE evaluation (range + 1/sqrt(n) weighting)
# Purpose: Compute the official-style wMAE using weights = (1/sqrt(n_samples)) / (value_range)
# Inputs expected in notebook state:
#   - train           : DataFrame with true targets (columns in `targets`) and same index as OOF preds indices
#   - targets         : list like ["Tg","FFV","Tc","Density","Rg"]
#   - oof_preds       : dict of per-target OOF Series (indexed by train rows where that target has labels)
#
# This cell builds a full oof DataFrame aligned to `train.index`, then evaluates per-target MAE and the weighted wMAE.

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# 1) Build an aligned OOF DataFrame from the dict (safe if already exists)
if isinstance(oof_preds, dict):
    oof_df = pd.DataFrame(index=train.index, columns=targets, dtype=float)
    for t in targets:
        if t in oof_preds and oof_preds[t] is not None:
            s = pd.to_numeric(oof_preds[t], errors="coerce")
            # only assign where we have model predictions
            oof_df.loc[s.index, t] = s.values
else:
    # assume it's already a DataFrame aligned to train.index
    oof_df = oof_preds.copy()
    # ensure numeric dtypes
    for t in targets:
        if t in oof_df.columns:
            oof_df[t] = pd.to_numeric(oof_df[t], errors="coerce")

# 2) Define the competition metric (as specified): weight = (1/sqrt(n)) / (value_range), normalized to sum=1
def competition_wmae(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame, targets_list):
    maes = {}
    weights = {}
    ns = {}  # sample counts per target (for sanity)

    for t in targets_list:
        if t not in y_true_df.columns or t not in y_pred_df.columns:
            continue

        y_true = pd.to_numeric(y_true_df[t], errors="coerce")
        y_pred = pd.to_numeric(y_pred_df[t], errors="coerce")

        mask = (~y_true.isna()) & (~y_pred.isna())
        n = int(mask.sum())
        if n == 0:
            continue

        y_t = y_true[mask].to_numpy()
        y_p = y_pred[mask].to_numpy()

        mae = mean_absolute_error(y_t, y_p)
        val_range = float(np.max(y_t) - np.min(y_t))
        if not np.isfinite(val_range) or val_range <= 0:
            val_range = 1.0  # guard against zero/degenerate range

        w = (1.0 / np.sqrt(n)) / val_range

        maes[t] = float(mae)
        weights[t] = float(w)
        ns[t] = n

    if not weights:
        return np.nan, maes, {}, ns

    total_w = sum(weights.values())
    norm_w = {t: w / total_w for t, w in weights.items()}
    wmae = sum(maes[t] * norm_w[t] for t in maes)

    return float(wmae), maes, norm_w, ns

# 3) Run metric on current OOF
y_true_df = train[targets].copy()
wmae_score, per_target_mae, norm_weights, counts = competition_wmae(y_true_df, oof_df, targets)

# 4) Print results
print("===== Per-target OOF MAE (competition components) =====")
for t in targets:
    if t in per_target_mae:
        w_str = f" | weight={norm_weights.get(t, 0):.6f}" if t in norm_weights else ""
        print(f"{t:8s}: MAE = {per_target_mae[t]:.5f} | n={counts.get(t, 0)}{w_str}")
    else:
        print(f"{t:8s}: (no evaluable rows)")

print("\n===== Overall Competition Metric (Corrected) =====")
print(f"wMAE (competition scale): {wmae_score:.5f}")

# 5) Optional: show naive averages for comparison (to confirm the old metric was different)
naive_mae_vals = [per_target_mae[t] for t in targets if t in per_target_mae]
if naive_mae_vals:
    print("\n[Diagnostics] Naive mean of MAEs:", f"{np.mean(naive_mae_vals):.5f}")
    # sample-size weighted average (NOT competition metric)
    sample_weighted = np.average(naive_mae_vals, weights=[counts[t] for t in targets if t in per_target_mae])
    print("[Diagnostics] Sample-size weighted MAE:", f"{sample_weighted:.5f}")

# 6) Sanity checks
if norm_weights:
    print("\nSanity: sum of normalized weights =", f"{sum(norm_weights.values()):.6f}")
    assert abs(sum(norm_weights.values()) - 1.0) < 1e-6, "Weights do not sum to 1."


===== Per-target OOF MAE (competition components) =====
Tg      : MAE = 10.79310 | n=510 | weight=0.000519
FFV     : MAE = 0.00135 | n=7019 | weight=0.157796
Tc      : MAE = 0.00710 | n=737 | weight=0.561011
Density : MAE = 0.01863 | n=613 | weight=0.268908
Rg      : MAE = 0.40495 | n=614 | weight=0.011766

===== Overall Competition Metric (Corrected) =====
wMAE (competition scale): 0.01957

[Diagnostics] Naive mean of MAEs: 2.24502
[Diagnostics] Sample-size weighted MAE: 0.60879

Sanity: sum of normalized weights = 1.000000


In [113]:
# %% Cell — Export submission (robust, aligned, no NaNs/inf)
import numpy as np, pandas as pd, hashlib, os

# ---- Preconditions & sanity checks ----
# Required objects: test, sample, targets, test_preds
missing = [name for name in ["test", "sample", "targets", "test_preds"] if name not in globals()]
assert not missing, f"Missing required object(s): {missing}"

assert isinstance(test, pd.DataFrame), "`test` must be a DataFrame"
assert isinstance(sample, pd.DataFrame), "`sample` must be a DataFrame"
assert isinstance(test_preds, pd.DataFrame), "`test_preds` must be a DataFrame"
assert isinstance(targets, (list, tuple)) and all(isinstance(t, str) for t in targets), "`targets` must be a list of strings"

# Ensure 'id' exists in test (preferred) or sample
id_source = None
if "id" in test.columns:
    id_source = test["id"].reset_index(drop=True)
elif "id" in sample.columns and len(sample) == len(test):
    id_source = sample["id"].reset_index(drop=True)
else:
    raise ValueError("Could not find an 'id' column in `test` (preferred) or a matching `sample`.")

# Ensure predictions contain all target columns
missing_cols = [c for c in targets if c not in test_preds.columns]
assert not missing_cols, f"`test_preds` is missing target column(s): {missing_cols}"

# Length check
assert len(test_preds) == len(test), f"Length mismatch: test_preds={len(test_preds)} vs test={len(test)}"

# ---- Build submission DataFrame in exact order ----
sub = pd.DataFrame({"id": id_source})
for col in targets:
    vals = test_preds[col].to_numpy()
    # Coerce numeric dtype and clean non-finite
    vals = vals.astype("float32", copy=False)
    if not np.isfinite(vals).all():
        bad = (~np.isfinite(vals)).sum()
        # Fill bad values with column median (or 0 if all non-finite)
        finite_vals = vals[np.isfinite(vals)]
        fill_val = float(np.median(finite_vals)) if finite_vals.size > 0 else 0.0
        print(f"[WARN] {col}: {bad} non-finite values in predictions → filling with {fill_val}")
        vals[~np.isfinite(vals)] = fill_val
    sub[col] = vals

# Enforce exact column order
expected_cols = ["id"] + list(targets)
sub = sub[expected_cols]

# Save
out_path = "/kaggle/working/submission.csv" if os.path.exists("/kaggle/working") else "submission.csv"
sub.to_csv(out_path, index=False)

# MD5 for reproducibility
md5 = hashlib.md5(open(out_path, "rb").read()).hexdigest()

print(f"✅ Submission saved: {out_path}")
print(f"MD5: {md5}")
display(sub.head())


✅ Submission saved: submission.csv
MD5: 528f6ca850f3675dffcc8b94f59f61d4


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,-1.246428,0.282082,0.113245,0.833577,11.573305
1,1422188626,5.537556,0.280498,0.114947,0.817431,11.412236
2,2032016830,-0.251643,0.283071,0.112197,0.815305,11.693082


In [114]:
# %% Cell — Verify submission format (strict)
import numpy as np, pandas as pd, os

out_path = "/kaggle/working/submission.csv" if os.path.exists("/kaggle/working") else "submission.csv"
check = pd.read_csv(out_path)

# 1) Columns & order
expected_cols = ["id"] + list(targets)
print("Columns in file:", list(check.columns))
assert list(check.columns) == expected_cols, f"❌ Column mismatch.\nExpected: {expected_cols}\nGot     : {list(check.columns)}"

# 2) Row count
print("Row count:", len(check), "| expected:", len(test))
assert len(check) == len(test), f"❌ Row count mismatch: got {len(check)}, expected {len(test)}"

# 3) id presence & uniqueness checks
assert "id" in check.columns, "❌ Missing 'id' column"
assert check["id"].isna().sum() == 0, "❌ 'id' column contains NaNs"
if "id" in test.columns:
    # only enforce equality if test has id column
    same_ids = (check["id"].reset_index(drop=True) == test["id"].reset_index(drop=True)).all()
    assert same_ids, "❌ 'id' column in submission does not match `test['id']` order"

# 4) Target columns: numeric, finite, no NaNs
for col in targets:
    assert np.issubdtype(check[col].dtype, np.number), f"❌ Column '{col}' is not numeric (dtype={check[col].dtype})"
    if not np.isfinite(check[col].to_numpy()).all():
        bad = (~np.isfinite(check[col].to_numpy())).sum()
        raise ValueError(f"❌ Column '{col}' has {bad} non-finite values")
    if check[col].isna().any():
        raise ValueError(f"❌ Column '{col}' contains NaNs")

# 5) Quick stats for sanity
print("\n=== Quick stats (targets) ===")
display(check[targets].describe())

print("\n✅ Submission file passes all checks and is ready for Kaggle!")


Columns in file: ['id', 'Tg', 'FFV', 'Tc', 'Density', 'Rg']
Row count: 3 | expected: 3

=== Quick stats (targets) ===


,Tg,FFV,Tc,Density,Rg
count,3.000000,3.000000,3.000000,3.000000,3.000000
mean,1.346495,0.281884,0.113463,0.822105,11.559541
std,3.663488,0.001298,0.001388,0.009992,0.140928
min,-1.246428,0.280498,0.112197,0.815305,11.412236
25%,-0.749035,0.281290,0.112721,0.816368,11.492770
50%,-0.251643,0.282082,0.113245,0.817431,11.573305
75%,2.642956,0.282577,0.114096,0.825504,11.633194
max,5.537556,0.283071,0.114947,0.833577,11.693082



✅ Submission file passes all checks and is ready for Kaggle!
